## This section involves implementing term-document retrieval techniques, including the term-document incidence matrix, inverted index, Jaccard similarity, and TF-IDF. The objective is to retrieve a set of recipes from a dataset based on a provided set of ingredients as a query.

## Author: Nam Dang

# Setting up the libraries and environment

In [178]:
%pip install pandas numpy scikit-learn nltk whoosh

43903.77s - pydevd: Sending message related to process being replaced timed-out after 5 seconds



[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [179]:
import pandas as pd
import numpy as np

# Data Pre-processing

In [180]:
df = pd.read_csv("./data/food_recipes.csv")
print(df.shape)
print(df.head())

(50000, 2)
                   title                                         directions
0    No-Bake Nut Cookies  In a heavy 2-quart saucepan, mix brown sugar, ...
1  Jewell Ball'S Chicken  Place chipped beef on bottom of baking dish., ...
2            Creamy Corn  In a slow cooker, combine all ingredients. Cov...
3          Chicken Funny  Boil and debone chicken., Put bite size pieces...
4   Reeses Cups(Candy)    Combine first four ingredients and press in 13...


In [181]:
# Step 1
df = df.dropna(subset=['title', 'directions'])

# Step 2
df['title'] = df['title'].str.lower()
df['directions'] = df['directions'].str.lower()

# Step 3
df['title'] = df['title'].str.strip()
df['directions'] = df['directions'].str.strip()

### Data Pre-processing Steps

1. Removed missing values:
   Rows with missing `title` or `directions` were removed to make sure the data is complete.

2. Converted text to lowercase:
   This ensures consistency so the same words with different capitalization are treated the same. 

3. Removed extra whitespace:
   This makes sure there is no extra whitespace to avoid unwanted errors

In [182]:
df.tail(5)

,title,directions
49995,caramel frosting,"let butter melt; add sugar and brown., add mil..."
49996,barbecued chicken wings,"mix sauce, brown sugar, onion and water, mixin..."
49997,pound cake,"bake at 275° for 1 1/2 hours., put toothpick o..."
49998,slush,"combine sugar and boiling water., stir and coo..."
49999,granny ebert's hash,"cover meat with water, 1 teaspoon of salt and ..."


# Term-Document Incidence Matrix


In [183]:
import re
import nltk

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [184]:
STOPWORDS = set(stopwords.words('english'))
lemmatiser = WordNetLemmatizer()

token_pattern = re.compile(r'[A-Za-z]{2,}')

def clean(text):
    tokens = token_pattern.findall(text.lower())
    tokens = [t for t in tokens if t not in STOPWORDS]
    tokens = [lemmatiser.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df['clean'] = df['directions'].apply(clean)

df[['directions', 'clean']].head()

,directions,clean
0,"in a heavy 2-quart saucepan, mix brown sugar, ...",heavy quart saucepan mix brown sugar nut evapo...
1,"place chipped beef on bottom of baking dish., ...",place chipped beef bottom baking dish place ch...
2,"in a slow cooker, combine all ingredients. cov...",slow cooker combine ingredient cover cook low ...
3,"boil and debone chicken., put bite size pieces...",boil debone chicken put bite size piece averag...
4,combine first four ingredients and press in 13...,combine first four ingredient press inch ungre...


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

bin_vectorizer = CountVectorizer(binary=True)

X_bin = bin_vectorizer.fit_transform(df['clean'])

In [ ]:
all_terms = set()

for doc in df['clean']:
    for word in doc.split():
        all_terms.add(word)

all_terms = sorted(list(all_terms))


matrix_barebone = []

for doc in df['clean']:

    words_in_doc = set(doc.split())

    row = []

    for term in all_terms:

        if term in words_in_doc:
            row.append(1)
        else:
            row.append(0)

    matrix_barebone.append(row)

# DataFrame
matrix_barebone_df = pd.DataFrame(
    matrix_barebone,
    columns=all_terms
)

In [ ]:
# Library Incidence Matrix
terms_lib = bin_vectorizer.get_feature_names_out()

matrix_lib_df = pd.DataFrame(
    X_bin.toarray(),
    columns=terms_lib
)

print("Library Incidence Matrix")
display(matrix_lib_df.head())

Library Incidence Matrix


,ability,able,ableskiver,absent,absolutely,absorb,absorbed,absorbent,absorbing,absorbs,...,zing,zip,ziploc,zipper,zippered,zita,ziti,zucchini,zuchini,zwieback
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Barebone Incidence Matrix

print("Barebone Incidence Matrix")
display(matrix_barebone_df.head())

Barebone Incidence Matrix


,ability,able,ableskiver,absent,absolutely,absorb,absorbed,absorbent,absorbing,absorbs,...,zing,zip,ziploc,zipper,zippered,zita,ziti,zucchini,zuchini,zwieback
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
def boolean_search(query, vectorizer, matrix):

    terms = query.split()
    stack = []
    vocab = vectorizer.vocabulary_

    for token in terms:

        if token in ('AND', 'OR', 'NOT'):
            stack.append(token)

        else:
            vec = np.zeros(matrix.shape[0], dtype=bool)

            if token.lower() in vocab:
                col = vocab[token.lower()]
                vec = matrix[:, col].toarray().ravel().astype(bool)

            stack.append(vec)

        while (
            len(stack) >= 3
            and isinstance(stack[-1], np.ndarray)
            and stack[-2] in ('AND', 'OR')
            and isinstance(stack[-3], np.ndarray)
        ):

            right = stack.pop()
            op = stack.pop()
            left = stack.pop()

            if op == 'AND':
                stack.append(left & right)
            else:
                stack.append(left | right)

    if stack and isinstance(stack[0], np.ndarray):
        return np.nonzero(stack[0])[0]

    return []

In [ ]:
# Library Search
query = "cheese AND milk"

library_results = boolean_search(
    query,
    bin_vectorizer,
    X_bin
)

print("Library Search Results:")
print(df.iloc[library_results]['title'].tolist())

Library Search Results:
['cheeseburger potato soup', 'pineapple nut pie', 'egg casserole', '*apricot salad', 'caramel pie', 'pizza casserole', 'egg cheese souffle', 'cream puff dessert', 'any flavor pan dessert', 'mexican casserole', 'bacon and egg breakfast bake', 'cherry pie', 'strawberry yum-yum', 'broccoli-cauliflower soup', 'peanut butter cream pie', 'cream tacos', 'baked cheese sandwiches', 'tater tot casserole', 'layered banana pineapple dessert', 'dirty pudding', 'macaroni pie', "thomas jefferson's macaroni and cheese pudding", 'broccoli almond', 'chicken, broccoli and cheese casserole', 'ragu cheese potatoes', 'cheese and chilies', 'fettucini with lemon vegetables', 'broccoli cheese soup', 'broccoli and rice casserole', 'red lobster cheese garlic biscuits', 'dirt pie', 'oreo dirt cake', 'squash souffle', 'cauliflower-broccoli casserole', 'swiss chicken casserole', 'quick swedish meatballs', 'broccoli casserole', "bertha's macaroni delight", 'light and dark', 'cheddar chowder',

In [ ]:
# Barebone Search
query_words = ["cheese", "milk"]

result = matrix_barebone_df.copy()

for word in query_words:

    if word in result.columns:
        result = result[result[word] == 1]
    else:
        result = result.iloc[0:0]

print("Barebone Search Results:")
print(df.loc[result.index, 'title'].tolist())

Barebone Search Results:
['cheeseburger potato soup', 'pineapple nut pie', 'egg casserole', '*apricot salad', 'caramel pie', 'pizza casserole', 'egg cheese souffle', 'cream puff dessert', 'any flavor pan dessert', 'mexican casserole', 'bacon and egg breakfast bake', 'cherry pie', 'strawberry yum-yum', 'broccoli-cauliflower soup', 'peanut butter cream pie', 'cream tacos', 'baked cheese sandwiches', 'tater tot casserole', 'layered banana pineapple dessert', 'dirty pudding', 'macaroni pie', "thomas jefferson's macaroni and cheese pudding", 'broccoli almond', 'chicken, broccoli and cheese casserole', 'ragu cheese potatoes', 'cheese and chilies', 'fettucini with lemon vegetables', 'broccoli cheese soup', 'broccoli and rice casserole', 'red lobster cheese garlic biscuits', 'dirt pie', 'oreo dirt cake', 'squash souffle', 'cauliflower-broccoli casserole', 'swiss chicken casserole', 'quick swedish meatballs', 'broccoli casserole', "bertha's macaroni delight", 'light and dark', 'cheddar chowder'

### Comparison

Both implementations returned similar search results because they were built from the same cleaned dataset.

### Library-Based Implementation
I Use CountVectorizer from Scikit-learn. It is faster and more efficient, and also requires less code.

### Barebone Implementation
I have to manually build the incidence matrix. However, I find it easier to understand how incidence matrices work by this way even though there are more code and processing time.

### Conclusion
Both implementations performed Boolean retrieval as I wanted and produced similar results. The library implementation seems to be better because it is faster, more efficient, and easier to use. On the other hand, the barebone implementation is useful for understanding the basic concept of an incidence matrix.

# Inverted Index


In [192]:
def map_function(document, doc_id):

    pairs = []

    words = document.split()

    for word in words:
        pairs.append((word, doc_id))

    return pairs

In [193]:
def reduce_function(mapped_data):

    inverted_index = {}

    for word, doc_id in mapped_data:

        if word not in inverted_index:
            inverted_index[word] = []

        if doc_id not in inverted_index[word]:
            inverted_index[word].append(doc_id)

    return inverted_index

In [194]:
def create_inverted_index_map_reduce(recipe_data):

    mapped_data = []

    for doc_id, document in enumerate(recipe_data['clean']):

        mapped_data.extend(
            map_function(document, doc_id)
        )

    inverted_index = reduce_function(mapped_data)

    return inverted_index

In [195]:
inverted_index = create_inverted_index_map_reduce(df)

print("Number of terms in inverted index:")
print(len(inverted_index))

Number of terms in inverted index:
6347


In [196]:
def inverted_index_search(recipe_data, inverted_index, search_terms):
    try:
        # Find all indices where all terms appear (boolean AND operation)
        sets_of_indices = [set(inverted_index[term]) for term in search_terms if term in inverted_index]
        if not sets_of_indices:
            return pd.DataFrame()  # Return an empty DataFrame if no indices found

        # Intersect all sets of indices to find common ones
        valid_indices = set.intersection(*sets_of_indices)

        # Convert the set of valid indices to a list
        valid_indices_list = list(valid_indices)

        # Select and return the relevant recipes using the valid indices
        return recipe_data.loc[valid_indices_list, ['title', 'directions']]
    except KeyError as e:
        # Handle the case where one or more search terms are not in the matrix
        print(f"Warning: {str(e).strip('[]')} not found in recipe terms.")
        return pd.DataFrame()  # Return an empty DataFrame if any term is not found

In [197]:
results = inverted_index_search(
    df,
    inverted_index,
    query_words
)

results

,title,directions
32769,ronald reagan's favorite macaroni and cheese,boil macaroni in water until tender and drain ...
5,cheeseburger potato soup,wash potatoes; prick several times with a fork...
16393,parmesan chicken(serves 8),dip each breast in a combination of eggs and m...
8202,striped delight,"combine crumbs, 1/4 cup sugar and melted butte..."
24589,upside-down pizza,"brown hamburger and onion., add salt., drain g..."
...,...,...
8178,overnight chicken casserole,"in a large bowl combine soup, milk and cheese...."
16373,chicken gumbo,"put bread slices in bottom of casserole dish.,..."
49142,chicken medley casserole(serves 6),"preheat oven to 350., cook noodles according t..."
16375,cheesy spicy spinach,cook spinach according to directions on packag...


# Inverted Index using Trees

In [198]:
# Reference: https://www.geeksforgeeks.org/python/trees-in-python/
class TreeNode:
    def __init__(self):
        self.postings = [] 


class InvertedIndexTree:
    def __init__(self):
        self.root = {} 

    def insert(self, term, doc_id):

        if term not in self.root:
            self.root[term] = TreeNode()

        if doc_id not in self.root[term].postings:
            self.root[term].postings.append(doc_id)

    def build(self, documents):

        for doc_id, doc in enumerate(documents):

            words = doc.split()

            for word in words:
                self.insert(word, doc_id)

        return self.root

In [199]:
tree_index = InvertedIndexTree()
tree_index.build(df['directions'])

print("Tree-based inverted index built.")
print("Number of terms:", len(tree_index.root))

Tree-based inverted index built.
Number of terms: 23731


In [200]:
def tree_search(tree_index, search_terms):

    results = None

    for term in search_terms:

        if term in tree_index.root:
            docs = set(tree_index.root[term].postings)
        else:
            return []

        if results is None:
            results = docs
        else:
            results = results.intersection(docs)

    return list(results)

In [201]:
results = tree_search(tree_index, query_words)

print("Tree Search Results:")
print(df.iloc[results][['title', 'directions']])

Tree Search Results:
                                             title  \
14338                             potato casserole   
8202                               striped delight   
18444                          caramel coconut pie   
36878                                   tamale pie   
45077                     pork chops with potatoes   
...                                            ...   
14324  onion dip for veggies or chips and pretzels   
16373                                chicken gumbo   
16375                         cheesy spicy spinach   
18424                           mexican taco salad   
28668                        egg-sausage casserole   

                                              directions  
14338  cook potatoes in jackets and dice., mix cream ...  
8202   combine crumbs, 1/4 cup sugar and melted butte...  
18444  brown coconut and pecans lightly in 1/4 cup ma...  
36878  brown hamburger and onion (and sausage if used...  
45077  brown pork chops., in a 2-qu

In [202]:
# Reference: https://www.geeksforgeeks.org/python/python-measure-time-taken-by-program-to-execute/

import time

times = []

for i in range(5):

    start = time.time()

    tree_search(tree_index, query_words)

    end = time.time()

    times.append(end - start)

avg_latency = sum(times) / len(times)

print("Average Latency (seconds):", avg_latency)

Average Latency (seconds): 0.0003499031066894531


# Jaccard Similarity


In [203]:
def jaccard_similarity(set1, set2):
    intersection = set1.intersection(set2)
    union = set1.union(set2)

    if len(union) == 0:
        return 0

    return len(intersection) / len(union)

def jaccard_similarity_search(recipe_data, inverted_index, search_terms):
    try:
        sets_of_indices = [set(inverted_index[term]) for term in search_terms if term in inverted_index]
        if not sets_of_indices:
            return pd.DataFrame() 
        valid_indices = set.intersection(*sets_of_indices)
        valid_indices_list = list(valid_indices)
        result = recipe_data.loc[valid_indices_list, ['title', 'directions']]
        result['jaccard_similarity'] = result['directions'].apply(lambda x: jaccard_similarity(set(x.split()), set(search_terms)))

        return result
    except KeyError as e:
        print(f"Warning: {str(e).strip('[]')} not found in recipe terms.")
        return pd.DataFrame() 

In [204]:

results = jaccard_similarity_search(df, inverted_index, query_words)

results

,title,directions,jaccard_similarity
32769,ronald reagan's favorite macaroni and cheese,boil macaroni in water until tender and drain ...,0.000000
5,cheeseburger potato soup,wash potatoes; prick several times with a fork...,0.000000
16393,parmesan chicken(serves 8),dip each breast in a combination of eggs and m...,0.012500
8202,striped delight,"combine crumbs, 1/4 cup sugar and melted butte...",0.034483
24589,upside-down pizza,"brown hamburger and onion., add salt., drain g...",0.016129
...,...,...,...
8178,overnight chicken casserole,"in a large bowl combine soup, milk and cheese....",0.017241
16373,chicken gumbo,"put bread slices in bottom of casserole dish.,...",0.048780
49142,chicken medley casserole(serves 6),"preheat oven to 350., cook noodles according t...",0.014085
16375,cheesy spicy spinach,cook spinach according to directions on packag...,0.037736


In [205]:
# Reference: https://www.geeksforgeeks.org/python/python-measure-time-taken-by-program-to-execute/

times = []

for i in range(5):

    start = time.time()

    jaccard_similarity_search(df, inverted_index, query_words)

    end = time.time()

    times.append(end - start)

avg_latency = sum(times) / len(times)

print("Average Jaccard Search Latency:", avg_latency)

Average Jaccard Search Latency: 0.012949419021606446


# TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = df['clean'].tolist()

tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

In [ ]:
def tfidf_search(query):

    query_vec = tfidf_vectorizer.transform([query])

    similarity_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    ranked_indices = similarity_scores.argsort()[::-1]

    top_results = ranked_indices[:10]

    return df.iloc[top_results][['title', 'directions']]

In [208]:
results = tfidf_search("cheese milk")

results

,title,directions
17769,baked macaroni and cheese,"in saucepan, melt margarine; stir in flour and..."
36301,scalloped potatoes,"cook potatoes with onion until tender., drain...."
24109,baked spaghetti and cheese,"boil spaghetti until cheese is cooked., melt v..."
28779,macaroni pie,"mix macaroni and cheese well., mix milk and eg..."
38452,lima bean delight,cook all ingredients but milk until beans are ...
1091,bertha's macaroni delight,"in saucepan, cook onions in margarine. stir in..."
47854,cheese soup,"saute until soft in the margarine the celery, ..."
9220,easy macaroni and cheese,"cook macaroni and drain., cut cheese into smal..."
27515,tuna and rice casserole,"mix all ingredients, except cheese in 9 x 11-i..."
34144,sausage quiche,"put 1/2 of cheese in cooled pie crust., sprink..."


In [209]:
# Reference: https://www.geeksforgeeks.org/python/python-measure-time-taken-by-program-to-execute/

times = []

for i in range(5):

    start = time.time()

    tfidf_search("cheese milk")

    end = time.time()

    times.append(end - start)

avg_latency = sum(times) / len(times)

print("Average TF-IDF Latency:", avg_latency)

Average TF-IDF Latency: 0.007008504867553711


# Search Index using Whoosh

In [210]:
# Using 10 percent of the data

df_small = df.sample(frac=0.1, random_state=42).reset_index(drop=True)

print(df_small.shape)

(5000, 3)


In [211]:
from whoosh.fields import Schema, TEXT
from whoosh.analysis import StemmingAnalyzer

schema = Schema(
    title=TEXT(stored=True),
    directions=TEXT(stored=True, analyzer=StemmingAnalyzer())
)

In [ ]:
import os
from whoosh import index

# Create index directory
index_dir = "whoosh_index"

if not os.path.exists(index_dir):
    os.mkdir(index_dir)

# Create index
ix = index.create_in(index_dir, schema)

# Add documents
writer = ix.writer()

for i, row in df_small.iterrows():
    writer.add_document(
        title=row['title'],
        directions=row['directions']
    )

writer.commit()

In [213]:
from whoosh.qparser import QueryParser
from whoosh.scoring import TF_IDF

In [214]:
def whoosh_search(query):

    results_list = []

    with ix.searcher(weighting=TF_IDF()) as searcher:

        query_parser = QueryParser("directions", ix.schema)
        q = query_parser.parse(query)

        results = searcher.search(q, limit=5)

        for r in results:
            results_list.append({
                "title": r["title"],
                "directions": r["directions"],
                "score": r.score
            })

    return results_list

I use Whoosh's TF-IDF scoring model because it balances term frequency and document rarity, making it suitable for recipe relevance ranking.

In [215]:
results = whoosh_search("cheese milk")

for r in results:
    print("Title:", r["title"])
    print("Score:", r["score"])
    print(r["directions"][:200], "...\n")

Title: baked spaghetti and cheese
Score: 19.017042954882413
boil spaghetti until cheese is cooked., melt velveeta cheese in 1 cup milk., blend mixture of cheese and milk and well drained spaghetti together., pour into casserole dish., sprinkle american cheese  ...

Title: scalloped potatoes
Score: 19.017042954882413
cook potatoes with onion until tender., drain., mix in salt, milk and cup of cheese (about 4 slices) until cheese melts., pour a layer of potato-milk mixture in a casserole dish., top with cheese slic ...

Title: chicken crescent rolls
Score: 16.16298245945698
mix chicken with 1/4 cup cheese., cook soup, milk and 1/4 cup cheese on stove until cheese is melted., place 1 tablespoon chicken mixture on wide end of each roll. roll it up., place soup mixture in t ...

Title: lemon cheese pie
Score: 11.031689776463487
combine crumbs and oleo., spray 9-inch pie plate with pam and press crumb mixture over back and sides of plate., bake at 350° for 10 minutes or until crisp., cool., 

In [216]:
# Reference: https://www.geeksforgeeks.org/python/python-measure-time-taken-by-program-to-execute/

# indexing time
index_times = []

for i in range(5):

    start = time.time()

    ix = index.create_in(index_dir, schema)

    writer = ix.writer()

    for i, row in df_small.iterrows():
        writer.add_document(title=row['title'], directions=row['directions'])

    writer.commit()

    end = time.time()

    index_times.append(end - start)

# search time 
search_times = []

for i in range(5):

    start = time.time()
    whoosh_search("cheese milk")
    end = time.time()

    search_times.append(end - start)

print("Avg indexing time:", sum(index_times)/len(index_times))
print("Avg search time:", sum(search_times)/len(search_times))

Avg indexing time: 1.732548475265503
Avg search time: 0.011213541030883789


In [217]:
def folder_size(path):
    total = 0

    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)

    return total / 1024  

size_kb = folder_size(index_dir)

print("Index size (KB):", size_kb)

Index size (KB): 3707.8447265625


The .seg file store the main storage component, while the .toc file stores metadata and structure of the index. There is also a write lock file which is used to prevent concurrent writes during indexing. Most of the disk space is used by the .seg files, which contain the inverted index structure. The total folder size represents the space complexity of the indexing system. 

# Performance Discussion

### Comparison of Service Latency

### 1. Incidence Matrix 
- Slowest among simple methods
- High time complexity due to matrix

### 2. Inverted Index 
- Directly maps terms to document IDs

### 3. Tree-Based Inverted Index 
- A bit overhead due to tree structure implementation

### 4. Jaccard Similarity 
- Requires set operations for every document

### 5. TF-IDF 
- More computation than Boolean retrieval methods

### 6. Whoosh 
- Fastest overall
- Optimized indexing and retrieval engine

### Comparison of Search Results

### 1. Incidence Matrix
- Returns exact Boolean matches only
- No ranking of results

### 2. Inverted Index (Tree + MapReduce)
- No ranking, only matching documents

### 3. Jaccard Similarity
- Improves results by ranking based on overlap

### 4. TF-IDF
- Provides ranked results based on term importance

### 5. Whoosh
- Best
- Combines indexing + ranking (TF-IDF scoring)